[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/implementation/notebooks/week02-solutions.ipynb)

> **Tip:** click the badge to open this notebook interactively in Google Colab and explore the cells live.

# Week 2 — Solutions & Live Exploration (Trainer Only)

Instructor-only companion to `docs/trainer/quiz-answers.md` and `docs/trainer/lab-solutions.md`.
**Do not distribute to learners** — it contains quiz answers and lab solutions.

Every code cell calls the repo's own `radar.*` functions — nothing is reimplemented — so the
answers are *computed*, not quoted, and you can perturb parameters mid-lesson to explore.

This is the **per-week companion** for Week 2 (roadmap stages 4–5). It captures the Week 2
quizzes (Doppler, Kalman), the Stage 4 recap (range-Doppler map), both Stage 4 stretches
(aliasing observation / slow-time windowing), Lab L2 (PRF doubling), the Stage 5 recap
(Kalman tracking), and the Stage 5 stretch (multi-target + Q/R mis-tuning) — plus a
terminology note on the overloaded word "channel". Run top-to-bottom; randomness is seeded
via `RadarConfig(seed=42)`.

## Setup

In [ ]:
import os
import subprocess
import sys

# Colab runs on a fresh VM each session, so clone the repo and install the
# package before `from radar import ...` works. Only runs in Colab (guarded
# so it never clones into a local checkout). Idempotent on re-run.
in_colab = "google.colab" in str(get_ipython().__class__)
if in_colab and not os.path.isdir("active-radar-tracker-basics"):
    subprocess.run(
        ["git", "clone", "-b", "implementation",
         "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"],
        check=True,
    )
if in_colab:
    os.chdir("active-radar-tracker-basics")
    subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
    repo_src = os.path.abspath("src")
    if repo_src not in sys.path:
        sys.path.insert(0, repo_src)  # prioritize local modules

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from radar import signal_gen, channel, receiver, doppler, viz
from radar.tracker import KalmanTracker, State
from radar.config import RadarConfig

cfg = RadarConfig(seed=42)
print(cfg)

## Concept check — what "channel" means here (and what it doesn't)

The word **channel** is overloaded in radar DSP, and Stage 4 is the first place
it can trip you up — so we pin it down once.

- **`channel.py` = the propagation channel.** This is the radio model of
  TX → target → RX: it applies the round-trip **delay**, the **attenuation**,
  the **noise**, and (now, opt-in via `apply_doppler=True`) the **Doppler phase
  rotation**. "Channel" here means *the physical path the signal travels*, not
  a bin of an FFT.
- **The FFT bins are NOT "channels."** This stage introduces the slow-time FFT.
  Its output has two axes: the **fast-time** axis (range → *range cells*) and
  the **slow-time** axis (Doppler → *Doppler cells / bins*). People sometimes
  loosely call a frequency bin a "channel," but in this codebase the
  range-Doppler map is just a 2-D array — we call its columns/rows *cells* or
  *bins*, never "channels."
- **Later stages reuse "channel" in new senses.** Stage 6+ model a *spatial*
  channel: an antenna array gives one received signal per element, and
  beamforming forms a *spatial channel* per look direction; adaptive nulling
  (Stage 10) steers a spatial null. So "channel" will mean: propagation (now),
  array-element path (Week 3), and spatial look-direction (Week 4). **The
  default meaning is the propagation channel in `channel.py`** unless a stage
  says otherwise.

Rule of thumb: `channel.` → propagation model; `fftshift`/`fft` → bins;
`array`/`beamformer` → spatial.

## Quiz 5 — Doppler shift

> `f_d = 2v/lambda` for the baseline 40 m/s target? (the theory behind the map)

In [ ]:
lam = 3e8 / cfg.fc_hz
fd = 2 * cfg.target_velocity_mps / lam
print(f"lambda = c/f_c = {lam:.5f} m")
print(f"f_d = 2v/lambda = 2*{cfg.target_velocity_mps}/{lam:.5f} = {fd:.1f} Hz")
print(f"(~ {fd/16.3:.0f} x the 16.3 Hz-per-m/s rule of thumb)")

## Quiz 6 — Velocity resolution & max unambiguous velocity

> `Delta_v` and `v_max` for the baseline CPI?

In [ ]:
delta_v = lam / (2 * cfg.n_pulses * cfg.pri_s)
v_max = lam / (4 * cfg.pri_s)
print(f"Delta_v = lambda/(2 N T) = {delta_v:.2f} m/s")
print(f"v_max   = lambda/(4 T)   = {v_max:.1f} m/s")

## Quiz 7 — Ambiguity

> Is the 40 m/s target ambiguous, and what does the map show?

In [ ]:
prf = 1 / cfg.pri_s
f_app = (fd + prf / 2) % prf - prf / 2   # folded slow-time frequency
v_app = f_app * lam / 2
print(f"40 m/s > v_max = {v_max:.1f} m/s  -> AMBIGUOUS")
print(f"f_d = {fd:.1f} Hz folds to {f_app:.1f} Hz within +/- {prf/2:.0f} Hz")
print(f"apparent velocity = {v_app:.1f} m/s (target reads as approaching)")

## Stage 4 recap — the range-Doppler map

Now we *see* the theory above in data. Stack the 64 matched-filter pulses and
FFT along **slow time** (pulse index). Each range bin becomes a Doppler
spectrum. The baseline 1000 m, 40 m/s target lands at its range cell; because
`40 m/s > v_max ≈ 30.6 m/s` it **aliases** to a folded velocity (~ −21 m/s),
which is the correct reading, not a bug.

In [ ]:
pulse = signal_gen.lfm_chirp(cfg)
tx = signal_gen.transmit_waveform(cfg)
tgt = channel.Target(range_m=1000.0, velocity_mps=40.0, snr_db=20.0)
rng = np.random.default_rng(cfg.seed)

# Stage 4 opts into the slow-time Doppler phase; Stages 1-3 leave it off.
rx = channel.simulate_channel(tx, [tgt], cfg, rng, apply_doppler=True)
mf = receiver.matched_filter(rx, pulse)
rd = doppler.range_doppler_map(mf, cfg)
v = doppler.velocity_axis(cfg)
r = doppler.range_axis(cfg)

r_idx = int(np.argmin(np.abs(r - 1000.0)))
d_idx = int(np.argmax(np.abs(rd[:, r_idx])))
print(f"RD map shape = {rd.shape}  (n_pulses, samples)")
print(f"target range cell = {r[r_idx]:.0f} m  (truth 1000 m)")
print(f"apparent velocity = {v[d_idx]:.1f} m/s  (aliased from 40 m/s)")
print(f"folded expectation ~ -21.2 m/s;  Delta_v = {v[1]-v[0]:.2f} m/s")

In [ ]:
fig = viz.plot_rd_map(rd, cfg)
plt.show()

## Stretch (Stage 4) — observe aliasing

The stretch (roadmap ◇) is to *raise `v` above `v_max` and observe/explain
aliasing*. The baseline 40 m/s is already above `v_max`, so sweep true velocity
and watch the **apparent** velocity fold at `±v_max` — a clean sawtooth. (No fix
here; Lab L2 below is the fix.)

In [ ]:
true_v = np.linspace(-60, 60, 25)
app_v = []
for vv in true_v:
    t = channel.Target(range_m=1000.0, velocity_mps=vv, snr_db=20.0)
    r_ = channel.simulate_channel(
        tx, [t], cfg, np.random.default_rng(cfg.seed), apply_doppler=True
    )
    m_ = receiver.matched_filter(r_, pulse)
    rd_ = doppler.range_doppler_map(m_, cfg)
    d_i = int(np.argmax(np.abs(rd_[:, r_idx])))
    app_v.append(v[d_i])
app_v = np.array(app_v)

fig, ax = plt.subplots()
ax.plot(true_v, app_v, "o-")
ax.plot([-v_max, v_max], [-v_max, v_max], "k--", label="truth")
ax.axhspan(-v_max, v_max, color="g", alpha=0.12, label="unambiguous window")
ax.set_xlabel("true velocity (m/s)")
ax.set_ylabel("apparent velocity (m/s)")
ax.set_title("Aliasing: apparent velocity folds at +/- v_max")
ax.legend()
plt.show()

## Lab L2 — PRF doubling (the fix for aliasing)

Separate hands-on lab (`docs/trainer/lab-solutions.md`): **double the PRF**
(PRI 1 ms → 0.5 ms) and rerun Doppler for the 40 m/s target. `v_max =
lambda/(4T)` doubles to ~61.2 m/s, so 40 m/s is no longer ambiguous — the peak
that folded to −21.2 m/s now appears at its true +40 m/s bin. Velocity
ambiguity is a slow-time *sampling* problem; raising PRF fixes it at the cost
of a shorter unambiguous range.

In [ ]:
cfg2 = RadarConfig(seed=42, pri_s=0.5e-3)
tx2 = signal_gen.transmit_waveform(cfg2)
rng2 = np.random.default_rng(cfg2.seed)
rx2 = channel.simulate_channel(tx2, [tgt], cfg2, rng2, apply_doppler=True)
mf2 = receiver.matched_filter(rx2, pulse)
rd2 = doppler.range_doppler_map(mf2, cfg2)
v2 = doppler.velocity_axis(cfg2)
r2 = doppler.range_axis(cfg2)

r_idx2 = int(np.argmin(np.abs(r2 - 1000.0)))
d_idx2 = int(np.argmax(np.abs(rd2[:, r_idx2])))
print(f"doubled PRF: v_max = {lam/(4*cfg2.pri_s):.1f} m/s (was {v_max:.1f})")
print(f"peak now at v = {v2[d_idx2]:.1f} m/s  -> true +40 m/s, no longer folded")
fig2 = viz.plot_rd_map(rd2, cfg2)
plt.show()

## Stretch (Stage 4) — Slow-time windowing

A bare slow-time FFT has Doppler sidelobes (~−13 dB, like any rectangular
window). Applying a Hamming window along slow time before the FFT trades
main-lobe width for much lower sidelobes — cleaner separation of nearby
velocities. Compare the Doppler slice through the target range cell.

In [ ]:
win = np.hamming(cfg.n_pulses)[:, None]      # (n_pulses, 1) -> broadcast over range
mf_w = mf * win
rd_w = doppler.range_doppler_map(mf_w, cfg)

def sidelobe_db(spec, peak_idx, guard=3):
    s = np.abs(spec).copy()
    lo, hi = max(0, peak_idx - guard), min(len(s), peak_idx + guard + 1)
    s[lo:hi] = 0.0
    return 20 * np.log10(s.max() / np.abs(spec).max())

sl_no = sidelobe_db(rd[:, r_idx], d_idx)
sl_win = sidelobe_db(rd_w[:, r_idx], d_idx)
print(f"sidelobe level (no window): {sl_no:.1f} dB")
print(f"sidelobe level (Hamming):   {sl_win:.1f} dB   (lower = cleaner Doppler)")

fig, ax = plt.subplots()
ax.plot(v, 20 * np.log10(np.abs(rd[:, r_idx]) + 1e-12), label="no window")
ax.plot(v, 20 * np.log10(np.abs(rd_w[:, r_idx]) + 1e-12), label="Hamming")
ax.axvline(v[d_idx], color="r", ls="--", lw=1)
ax.set_xlabel("velocity (m/s)")
ax.set_ylabel("Doppler slice (dB)")
ax.set_title("Slow-time windowing: sidelobes vs mainlobe")
ax.legend()
plt.show()

## Quiz 8 — Kalman Q and R

> What happens if `Q` or `R` is mis-tuned?

`Q` = process noise (how wrong the constant-velocity model can be); `R` =
measurement noise (how noisy the detections are). The Kalman gain `K` is the
relative-trust dial:

- **`Q` too big** → the filter trusts the *sensor* → the track **jitters**
  (follows the noise).
- **`R` too big** → the filter trusts the *model* → the track is **sluggish /
  lags**, especially through a transient.

Watch it: run the same detections through three tunings and compare the tracked
velocity.

In [ ]:
dt, n, r0, vel, sigma = 0.1, 80, 1000.0, 40.0, 5.0
rng = np.random.default_rng(cfg.seed)
meas = [
    receiver.Detection(
        r0 + vel * k * dt + rng.normal(0, sigma),
        vel + rng.normal(0, sigma),
    )
    for k in range(n)
]

def run(q, r):
    t = KalmanTracker(dt, q, r)
    for m in meas:
        t.predict(); t.update(m)
    return t.track

tr_jitter = run(1e3, sigma**2)   # huge Q  -> trusts sensor
tr_ok     = run(2.0, sigma**2)    # nominal
tr_slug   = run(2.0, 1e4)         # huge R  -> trusts model

def vel_std(track):  return np.std([s.velocity_mps for s in track[15:]])
def vel_lag(track):  return np.mean([s.velocity_mps - vel for s in track[:15]])
print(f"Q huge -> tracked-velocity std = {vel_std(tr_jitter):.2f} m/s  (jittery)")
print(f"Q nominal -> tracked-velocity std = {vel_std(tr_ok):.2f} m/s")
print(f"R huge -> start-up velocity lag = {vel_lag(tr_slug):.2f} m/s  (sluggish)")

## Stage 5 recap — Kalman tracking

Each CPI gives a noisy `Detection` (here modeled as truth ± σ, σ = 5 m / 5 m/s
— i.e. the sensor's per-CPI output). The `KalmanTracker` blends those with the
constant-velocity model. The track is visibly smoother than the dots, and its
steady-state RMS error sits **below** the measurement noise.

In [ ]:
dt, n, r0, vel, sigma = 0.1, 80, 1000.0, 40.0, 5.0
rng = np.random.default_rng(cfg.seed)
true_states, meas = [], []
for k in range(n):
    R = r0 + vel * k * dt
    true_states.append(State(R, vel))
    meas.append(receiver.Detection(
        R + rng.normal(0, sigma), vel + rng.normal(0, sigma)))

tr = KalmanTracker(dt, q=2.0, r=sigma**2)
for m in meas:
    tr.predict(); tr.update(m)

ta = np.array([[s.range_m, s.velocity_mps] for s in tr.track])
ra = np.array([[s.range_m, s.velocity_mps] for s in true_states])
k0 = 15
rms = np.sqrt(np.mean((ta[k0:] - ra[k0:]) ** 2, axis=0))
print(f"steady-state RMS range    = {rms[0]:.2f} m   (measurement noise {sigma} m)")
print(f"steady-state RMS velocity = {rms[1]:.2f} m/s (measurement noise {sigma} m/s)")
print("-> track error is below the sensor noise: the filter earns its keep")

fig = viz.plot_track(true_states, meas, tr.track)
plt.show()

## Stretch (Stage 5) — Multi-target + Q/R mis-tuning

Two targets at **(400 m, +15 m/s)** and **(1200 m, −20 m/s)** → two *independent*
`KalmanTracker`s, one fed by each target's detection stream. (Data association —
deciding which detection belongs to which track — is deliberately out of scope.)
Below, each track recovers its own range/velocity trend from the noisy dots.

In [ ]:
dt, n = 0.1, 80
targets = [(400.0, 15.0), (1200.0, -20.0)]
rng = np.random.default_rng(cfg.seed)
truths, streams = [], []
for R0, vv in targets:
    tru, me = [], []
    for k in range(n):
        R = R0 + vv * k * dt
        tru.append(State(R, vv))
        me.append(receiver.Detection(R + rng.normal(0, 5), vv + rng.normal(0, 5)))
    truths.append(tru); streams.append(me)

tracks = []
for me in streams:
    t = KalmanTracker(dt, 2.0, 25.0)
    for m in me:
        t.predict(); t.update(m)
    tracks.append(t.track)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
for tru, tk in zip(truths, tracks):
    axes[0].plot([s.range_m for s in tru], label=f"true {tru[0].velocity_mps:+.0f} m/s")
    axes[0].plot([s.range_m for s in tk], "--")
    axes[1].plot([s.velocity_mps for s in tru])
    axes[1].plot([s.velocity_mps for s in tk], "--")
axes[0].set_ylabel("range (m)"); axes[1].set_ylabel("velocity (m/s)")
axes[1].set_xlabel("CPI index")
axes[0].set_title("Stretch: two independent tracks")
axes[0].legend(); fig.tight_layout()
plt.show()

## Extension (challenge) — tracking from live pipeline detections

> **Optional / advanced.** So far every measurement was modeled as truth ± σ.
> A more faithful — and harder — exercise is to generate each CPI's detection
> from the **actual radar pipeline** (Stages 1–4): simulate the propagation
> channel, matched-filter, build the range-Doppler map, then read the target's
> `(range, velocity)` off the map's peak. Feed *those* detections to the same
> `KalmanTracker` and compare. Discussion points for strong learners:
> - range is quantized to ~7.5 m bins; velocity to `Δv ≈ 0.96 m/s` bins;
> - any target with `|v| > v_max` **aliases** (try `vel = 40.0`);
> - the detection can occasionally be a noise/false peak — how does the tracker
>   cope? (this cell has a non-aliased `vel = 15.0` so the track is clean)
>
> Note: at this SNR the Doppler peak lands on the **same bin every CPI**, so the
> velocity reading is stable (accurate to its `Δv` bin) — the visible measurement
> noise, and the Kalman's payoff, is in **range** (±7.5 m bins). Lower the SNR
> (`snr_db = 5`) to make both range *and* velocity jittery.

In [ ]:
# Challenge: one real CPI per measurement, detection read off the RD-map peak.
K = 30
dt_cpi = cfg.n_pulses * cfg.pri_s           # seconds per CPI (back-to-back)
R0, vel = 1000.0, 15.0                       # non-aliased -> clean track
pulse = signal_gen.lfm_chirp(cfg)
tx = signal_gen.transmit_waveform(cfg)
r_axis = doppler.range_axis(cfg)
v_axis = doppler.velocity_axis(cfg)

true_states, meas_pipe = [], []
for k in range(K):
    Rk = R0 + vel * k * dt_cpi
    tgt = channel.Target(range_m=Rk, velocity_mps=vel, snr_db=20.0)
    rng_k = np.random.default_rng(cfg.seed + k)
    rx = channel.simulate_channel(tx, [tgt], cfg, rng_k, apply_doppler=True)
    mf = receiver.matched_filter(rx, pulse)
    rd = doppler.range_doppler_map(mf, cfg)
    di, ri = np.unravel_index(np.argmax(np.abs(rd)), rd.shape)  # (doppler, range)
    meas_pipe.append(receiver.Detection(range_m=r_axis[ri], velocity=v_axis[di]))
    true_states.append(State(Rk, vel))

tr_pipe = KalmanTracker(dt_cpi, q=2.0, r=25.0)
for m in meas_pipe:
    tr_pipe.predict(); tr_pipe.update(m)

ta = np.array([[s.range_m, s.velocity_mps] for s in tr_pipe.track])
ra = np.array([[s.range_m, s.velocity_mps] for s in true_states])
mp = np.array([[m.range_m, m.velocity] for m in meas_pipe])
k0 = 5
print(f"live-detection range jitter    = {np.std(mp[k0:, 0] - ra[k0:, 0]):.2f} m")
print(f"live-detection velocity jitter = {np.std(mp[k0:, 1] - ra[k0:, 1]):.2f} m/s")
print(f"tracked RMS range    = {np.sqrt(np.mean((ta[k0:] - ra[k0:]) ** 2, axis=0))[0]:.2f} m")
print(f"tracked RMS velocity = {np.sqrt(np.mean((ta[k0:] - ra[k0:]) ** 2, axis=0))[1]:.2f} m/s")

fig = viz.plot_track(true_states, meas_pipe, tr_pipe.track)
plt.show()

## Free play

Change **one** parameter, re-run the relevant cell, and explain the effect.
Seeded, so results are reproducible:

- `cfg.pri_s = 0.5e-3` — `v_max` doubles; 40 m/s unfolds (see Lab L2 above).
- `cfg.target_velocity_mps = 15.0` — peak at true +15 m/s (unambiguous).
- `cfg.n_pulses = 128` — `Delta_v` halves (finer velocity resolution).
- In Stage 5: set `KalmanTracker(dt, q=1e3, r=25)` to watch the track **jitter**,
  or `KalmanTracker(dt, 2.0, r=1e4)` to watch it **lag** the transient.
- Add a 2nd `Target` at `(5000 m, -20 m/s)` to the Stage 4 `simulate_channel`
  call → two blobs in the range-Doppler map.